In [12]:
# imports

import pandas as pd
import numpy as np
import json
import re
import string

In [2]:
# Helpers to read in data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, entailed_status, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, entailed_status in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [3]:
# Load in data frames

df_full_cleaned = load_and_convert_data_to_dataframe("data/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['entailed_status'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['entailed_status'] == 1]

df_r1_training = load_and_convert_data_to_dataframe("data/r1_training_all.json")

df_r2_training = load_and_convert_data_to_dataframe("data/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe("data/train_examples.json")

df_test = load_and_convert_data_to_dataframe("data/test_examples.json")

df_validation = load_and_convert_data_to_dataframe("data/val_examples.json")


In [4]:
# Replicate their simple and complex channel data analysis: gathering

# Identify unique tables
tables_in_full_cleaned = df_full_cleaned["table_id"].unique()
tables_in_r1 = df_r1_training["table_id"].unique()
tables_in_r2 = df_r2_training["table_id"].unique()

# Find tables that are in r1/r2 and in full cleaned
tables_in_r1_and_full_cleaned = np.intersect1d(tables_in_r1, tables_in_full_cleaned)
tables_in_r2_and_full_cleaned = np.intersect1d(tables_in_r2, tables_in_full_cleaned)

# Identify r1 and r2 statements in full cleaned
r1_in_full_cleaned = df_full_cleaned[df_full_cleaned["table_id"].isin(tables_in_r1_and_full_cleaned)]
r2_in_full_cleaned = df_full_cleaned[df_full_cleaned["table_id"].isin(tables_in_r2_and_full_cleaned)]

# Average length of entailed statements
full_cleaned_entailed_statements = df_full_cleaned_entailed["statement"]
average_length_full_cleaned_entailed_statements = full_cleaned_entailed_statements.str.split().apply(len).mean()

full_cleaned_entailed_statements_from_r1_tables = r1_in_full_cleaned[r1_in_full_cleaned["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements_from_r1_tables = full_cleaned_entailed_statements_from_r1_tables.str.split().apply(len).mean()

full_cleaned_entailed_statements_from_r2_tables = r2_in_full_cleaned[r2_in_full_cleaned["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements_from_r2_tables = full_cleaned_entailed_statements_from_r2_tables.str.split().apply(len).mean()


# Average length of refuted statements
full_cleaned_refuted_statements = df_full_cleaned_refuted["statement"]
average_length_full_cleaned_refuted_statements = full_cleaned_refuted_statements.str.split().apply(len).mean()

full_cleaned_refuted_statements_from_r1_tables = r1_in_full_cleaned[r1_in_full_cleaned["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements_from_r1_tables = full_cleaned_refuted_statements_from_r1_tables.str.split().apply(len).mean()

full_cleaned_refuted_statements_from_r2_tables = r2_in_full_cleaned[r2_in_full_cleaned["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements_from_r2_tables = full_cleaned_refuted_statements_from_r2_tables.str.split().apply(len).mean()

In [6]:
# Replicate their simple and complex channel data analysis: displaying results 

# Tables
print(f"Number of unique tables in full cleaned dataset: {len(tables_in_full_cleaned)}")
print(f"Number of unique tables in r1 training dataset: {len(tables_in_r1)}")
print(f"Number of unique tables in r2 training dataset: {len(tables_in_r2)}")

# Number of statements
print(f"Number of statements in full cleaned dataset: {len(df_full_cleaned)}")
print(f"Number of statements in r1 training dataset: {len(r1_in_full_cleaned)}")
print(f"Number of statements in r2 training dataset: {len(r2_in_full_cleaned)}")

# Statement length of entailed statements
print(f"Average length of entailed statements in full cleaned dataset: {average_length_full_cleaned_entailed_statements:.2f} words")
print(f"Average length of entailed statements in r1 training dataset: {average_length_full_cleaned_entailed_statements_from_r1_tables:.2f} words")
print(f"Average length of entailed statements in r2 training dataset: {average_length_full_cleaned_entailed_statements_from_r2_tables:.2f} words")

# Statement length of refuted statements
print(f"Average length of refuted statements in full cleaned dataset: {average_length_full_cleaned_refuted_statements:.2f} words")
print(f"Average length of refuted statements in r1 training dataset: {average_length_full_cleaned_refuted_statements_from_r1_tables:.2f} words")
print(f"Average length of refuted statements in r2 training dataset: {average_length_full_cleaned_refuted_statements_from_r2_tables:.2f} words")


Number of unique tables in full cleaned dataset: 16573
Number of unique tables in r1 training dataset: 9181
Number of unique tables in r2 training dataset: 7392
Number of statements in full cleaned dataset: 117854
Number of statements in r1 training dataset: 49850
Number of statements in r2 training dataset: 68004
Average length of entailed statements in full cleaned dataset: 13.71 words
Average length of entailed statements in r1 training dataset: 13.05 words
Average length of entailed statements in r2 training dataset: 14.19 words
Average length of refuted statements in full cleaned dataset: 13.79 words
Average length of refuted statements in r1 training dataset: 13.32 words
Average length of refuted statements in r2 training dataset: 14.15 words


In [7]:
# Checking number of satetements in training, test, and validation datasets

print(f"Number of statements in training dataset: {len(df_training)}")
print(f"Number of statements in test dataset: {len(df_test)}")
print(f"Number of statements in validation dataset: {len(df_validation)}")
print(f"Total number of statements in training, test, and validation datasets: {len(df_training) + len(df_test) + len(df_validation)}")


Number of statements in training dataset: 92283
Number of statements in test dataset: 12779
Number of statements in validation dataset: 12792
Total number of statements in training, test, and validation datasets: 117854


In [8]:
# Entity analysis

# Counting average ENT tags per statement
average_entities_per_statement_full_cleaned = df_full_cleaned['pos_tags'].str.count(r'\bENT\b').mean()
average_entities_per_entailed_statement = df_full_cleaned_entailed['pos_tags'].str.count(r'\bENT\b').mean()
average_entities_per_refuted_statement = df_full_cleaned_refuted['pos_tags'].str.count(r'\bENT\b').mean()

print(f"Average number of ENT tags per statement in full cleaned dataset: {average_entities_per_statement_full_cleaned:.2f}")
print(f"Average number of ENT tags per entailed statement: {average_entities_per_entailed_statement:.2f}")
print(f"Average number of ENT tags per refuted statement: {average_entities_per_refuted_statement:.2f}")

# Counting average content words
content_word_regex = r'\b(ENT|NN|NNS|NNP|NNPS|VB|VBD|VBG|VBN|VBP|VBZ|JJ|JJR|JJS|RB|RBR|RBS|CD)\b'
average_content_words_per_statement_full_cleaned = df_full_cleaned['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_entailed_statement = df_full_cleaned_entailed['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_refuted_statement = df_full_cleaned_refuted['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_statement_r1 = r1_in_full_cleaned['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_statement_r2 = r2_in_full_cleaned['pos_tags'].str.count(content_word_regex).mean()

print(f"Average number of content words per statement in full cleaned dataset: {average_content_words_per_statement_full_cleaned:.2f}")
print(f"Average number of content words per entailed statement: {average_content_words_per_entailed_statement:.2f}")
print(f"Average number of content words per refuted statement: {average_content_words_per_refuted_statement:.2f}")
print(f"Average number of content words per statement in r1 training dataset: {average_content_words_per_statement_r1:.2f}")
print(f"Average number of content words per statement in r2 training dataset: {average_content_words_per_statement_r2:.2f}")

Average number of ENT tags per statement in full cleaned dataset: 3.49
Average number of ENT tags per entailed statement: 3.51
Average number of ENT tags per refuted statement: 3.46
Average number of content words per statement in full cleaned dataset: 7.15
Average number of content words per entailed statement: 7.08
Average number of content words per refuted statement: 7.23
Average number of content words per statement in r1 training dataset: 6.50
Average number of content words per statement in r2 training dataset: 7.62


In [48]:
# Calculating word counts

# Kind of useless
def get_non_entity_statement_word_counts(df):
    statements_without_entities = (df['statement']
                                   .str.replace(r'#.*?#', '', regex=True) # get rid of hashtags
                                   .str.lower()
                                   .str.replace(rf'[{re.escape(string.punctuation)}]', ' ', regex=True)) # get rid of punctuation
    word_counts = statements_without_entities.str.split().explode().value_counts() # get words into single list of words and count them
    return word_counts

# More useful
def get_non_entity_statement_content_words(df):
    statements_without_entities = (df['statement']
                                   .str.replace(r'#.*?#', '', regex=True) # get rid of hashtags
                                   .str.lower()
                                   .str.split())
    pos_tags_without_entities = df['pos_tags'].str.replace(r'\bENT\b', '', regex=True).str.split() # get rid of hashtags
    for statement_list, pos_tag_list in zip(statements_without_entities, pos_tags_without_entities):
        content_words = [word for word, pos in zip(statement_list, pos_tag_list) if re.match(content_word_regex, pos)]
        yield from content_words

full_cleaned_content_word_counts = Counter(get_non_entity_statement_content_words(df_full_cleaned))
print(f"Most common content words in full cleaned dataset: {full_cleaned_content_word_counts.most_common(10)}")

full_cleaned_entailed_content_word_counts = Counter(get_non_entity_statement_content_words(df_full_cleaned_entailed))
print(f"Most common content words in entailed statements: {full_cleaned_entailed_content_word_counts.most_common(10)}")

full_cleaned_refuted_content_word_counts = Counter(get_non_entity_statement_content_words(df_full_cleaned_refuted))
print(f"Most common content words in refuted statements: {full_cleaned_refuted_content_word_counts.most_common(10)}")

Most common content words in full cleaned dataset: [('be', 78143), ('have', 28405), ('game', 15298), ('play', 10075), ('only', 8451), ('2', 8051), ('1', 7684), ('more', 7546), ('time', 6665), ('win', 6661)]
Most common content words in entailed statements: [('be', 41481), ('have', 15391), ('game', 8260), ('play', 5389), ('only', 4523), ('2', 4467), ('1', 4226), ('win', 3609), ('time', 3597), ('more', 3595)]
Most common content words in refuted statements: [('be', 36662), ('have', 13014), ('game', 7038), ('play', 4686), ('more', 3951), ('only', 3928), ('2', 3584), ('1', 3458), ('time', 3068), ('win', 3052)]
